# Notebook B — Offline Policy Training (Stage A) + GNN Fine-tune (Stage B, optional)

## Setup (carried over from your already-run Notebook A)
These 6 cells are copied verbatim from the Notebook A you already ran successfully
(same PAT-based private clone, same `INPUT` dataset path, same config, same frozen
model checkpoint, same inline RL helper classes). You do **not** need to re-run
Notebook A first in this session — just run these cells here, top to bottom, then
continue into Stage A training below.

**One cleanup applied:** the old dead public `git clone` line in the "mount data"
cell has been removed — your Cell 2 (PAT clone) already creates the repo folder, so
that line never actually ran anyway; this just removes the leftover.

## Setup (from your already-run Notebook A)

In [ ]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
# Audited implementation lives on fork feature/rl-pipeline (upstream main
# lags the audited commits); pin both so execution matches the reviewed code.
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "feature/rl-pipeline"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            "--branch", REPO_BRANCH,
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

os.chdir(REPO_DIR)
print(os.listdir("."))

In [ ]:
# Mount uploaded data (repo already cloned privately in the previous cell)
import subprocess, os, shutil
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")


In [ ]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

In [ ]:
# PROVENANCE (recording only - no computation is affected; no secrets are read)
import hashlib, json, platform, subprocess, time

def _md5_of(path, _blk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(_blk), b""):
            h.update(chunk)
    return h.hexdigest()

def _file_info(path):
    import os
    ex = os.path.exists(path)
    return {"path": path, "exists": ex,
            "size_bytes": os.path.getsize(path) if ex else None,
            "md5": _md5_of(path) if ex else None}

def _git_info(*args):
    try:
        return subprocess.check_output(["git", "-C", REPO_DIR, *args],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

def record_provenance(stage, extra=None):
    """Append one stage entry to outputs/rls/provenance.json (keyed by stage,
    so A/B/B_policy/C/D entries coexist and are never silently overwritten)."""
    import os
    entry = {
        "stage": stage,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "repo": {"head": _git_info("rev-parse", "HEAD"),
                 "branch": _git_info("rev-parse", "--abbrev-ref", "HEAD"),
                 "describe": _git_info("describe", "--always")},
        "dataset": {"tng_csv": _file_info("data/raw/tng100_clustered.csv"),
                    "checkpoint": _file_info(f"{INPUT}/best_model_augmented.pt")},
        "seed": cfg.get("seed"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": {"available": torch.cuda.is_available(),
                 "version": str(torch.version.cuda),
                 "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "config_used": {"data": cfg.get("data"), "graph": cfg.get("graph"),
                        "model": cfg.get("model"), "rls": cfg.get("rls")},
    }
    if extra:
        entry.update(extra)
    os.makedirs("outputs/rls", exist_ok=True)
    prov = {}
    try:
        with open("outputs/rls/provenance.json") as f:
            prov = json.load(f)
    except Exception:
        pass
    prov[stage] = entry
    with open("outputs/rls/provenance.json", "w") as f:
        json.dump(prov, f, indent=2)
    print(f"[provenance] stage '{stage}' -> outputs/rls/provenance.json "
          f"(HEAD={entry['repo']['head']}, csv md5={entry['dataset']['tng_csv']['md5']})")

record_provenance("B")

In [ ]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

In [ ]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

In [ ]:
# Inline RL helpers (policy net, sparsify, reward) — matches rls/ package
# PREFER: from rls.policy import EdgePolicyNet
# PREFER: from rls.sparsify import hard_mask, repair_connectivity
# PREFER: from rls.policy_gradient import bernoulli_logp
# (swap in the above if rls/ is importable — see TIP at the top of this file)
import torch
import torch.nn as nn
import torch.nn.functional as F

class EdgePolicyNet(nn.Module):
    def __init__(self, edge_dim=5, node_emb_dim=128, hidden_dim=64):
        super().__init__()
        self.node_proj = nn.Linear(node_emb_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*4, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, 1))
    def forward(self, edge_attr, node_emb, edge_index, context):
        e = F.leaky_relu(self.edge_proj(edge_attr), 0.1)
        u, v = edge_index
        nu = F.leaky_relu(self.node_proj(node_emb[u]), 0.1)
        nv = F.leaky_relu(self.node_proj(node_emb[v]), 0.1)
        c = context.unsqueeze(0).expand(e.size(0), -1)
        return self.fc(torch.cat([nu, nv, e, c], dim=-1))

def hard_mask(probs, min_keep_frac=0.1):
    # BOOL mask: edge_index[:, mask] / edge_attr[mask] require bool; an int
    # 0/1 mask silently positional-indexes (duplicates edges 0 and 1).
    mask = (probs >= 0.5)
    k = int(torch.ceil(torch.tensor(min_keep_frac) * probs.numel()))
    if mask.sum() < k:
        mask = torch.zeros_like(mask, dtype=torch.bool)
        mask[torch.topk(probs, k).indices] = True
    return mask

def apply_min_keep_floor(mask, probs, min_keep_frac=0.1):
    """Same floor-enforcement as hard_mask, but the candidate mask need
    not be a thresholded-probs mask — e.g. a sampled Bernoulli action.
    If the candidate keeps fewer than min_keep_frac of edges, replace it
    with the top-(min_keep_frac) highest-probability edges instead."""
    assert mask.dtype == torch.bool, f"expected bool mask, got {mask.dtype}"
    mask = mask.clone()
    k_min = int(torch.ceil(torch.tensor(min_keep_frac) * probs.numel()))
    if mask.sum() < k_min:
        top = torch.topk(probs, k_min).indices
        mask = torch.zeros_like(mask, dtype=torch.bool)
        mask[top] = True
    return mask

def repair_connectivity(edge_index, mask):
    mask = mask.clone()
    kept = mask.bool()
    incident = torch.zeros(edge_index.max().item()+1, dtype=torch.long)
    for i in range(edge_index.shape[1]):
        if kept[i]:
            incident[edge_index[0, i]] += 1; incident[edge_index[1, i]] += 1
    for node in (incident == 0).nonzero(as_tuple=True)[0].tolist():
        cand = (edge_index == node).sum(dim=0).bool()
        if cand.any():
            mask[cand.nonzero(as_tuple=True)[0][0]] = True
    return mask

def bernoulli_logp(p, action, eps=1e-8):
    return torch.where(action.bool(), torch.log(p.clamp(eps, 1.0)),
                       torch.log((1-p).clamp(eps, 1.0)))

def virial_ratio_pruned(ke, pe, eps=1e-8):
    return (2.0 * ke) / torch.abs(pe).clamp(min=eps)

## Stage A: precompute embeddings, then train the RL policy

In [ ]:
# CELL 6: Precompute frozen node embeddings + graph contexts for ALL graphs
from torch_geometric.nn import global_mean_pool
import torch_geometric.data as pg
import pandas as pd   # used by the training-log save in Cell 10

gnn.eval()
def prepare(loader):
    out = []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            emb = gnn.get_embeddings(b, embedding_point="pre_pooling")   # [N, out]
            ctx = global_mean_pool(emb, b.batch)                          # [B, out]
            for i in range(b.num_graphs):
                g = b.get_example(i)
                n = g.x.shape[0]
                out.append({
                    "x": g.x, "edge_index": g.edge_index, "edge_attr": g.edge_attr,
                    "y": g.y, "ctx": ctx[i], "emb": emb[b.batch == i],
                    "stellar_mass": g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(n)*1e10,
                    "vel_disp": g.vel_disp if hasattr(g, "vel_disp") else torch.ones(n)*100,
                    "half_mass_r": g.half_mass_r if hasattr(g, "half_mass_r") else torch.ones(n)*0.01,
                    "pos": g.pos if hasattr(g, "pos") else torch.zeros(n, 3),
                })
    return out

train_graphs = prepare(train_loader)
val_graphs = prepare(val_loader)
test_graphs = prepare(test_loader)
print("train graphs:", len(train_graphs), "| ctx dim:", train_graphs[0]["ctx"].shape)


In [ ]:
# CELL 7: Policy-gradient helpers (REINFORCE + learned baseline — honestly NOT PPO:
# one-step MDP => no GAE, no importance ratio; advantage = reward - baseline)
class ValueNet(nn.Module):
    def __init__(self, emb_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(emb_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, 1))
    def forward(self, ctx): return self.net(ctx).squeeze(-1)

def compute_advantages(rewards, values):
    return rewards - values.detach()

def bernoulli_entropy(p, eps=1e-6):
    # eps must be float32-representable: 1 - 1e-8 rounds to exactly 1.0 in
    # float32, making the upper clamp a no-op -> 0*log(0) = NaN for saturated
    # probs (sigmoid(x)>=17 saturates). Mirrors rls/policy_gradient.py and its
    # regression test test_entropy_and_logp_finite_at_saturated_probs.
    p = p.clamp(eps, 1.0 - eps)
    return -(p * torch.log(p) + (1 - p) * torch.log(1 - p))


In [ ]:
# CELL 8: Graph-physics terms (virial penalty) + reward
def graph_physics_terms(g, mask, G=4.302e-9):
    # PHYSICS EDGES EXCLUDE SELF-LOOPS (approved relative-virial fix, Aug 2026):
    # a self-pair's r==0 clamps to 1e-6 and dominated PE (~100% on real graphs),
    # making the old one-sided penalty identically zero. Physics-only filter;
    # GNN message passing keeps its self-loops.
    real = g["edge_index"][0] != g["edge_index"][1]
    edge_index = g["edge_index"][:, real]; mask = mask.to(device)[real]
    stellar = g["stellar_mass"].to(device); vd = g["vel_disp"].to(device); pos = g["pos"].to(device)
    deg = torch.zeros(stellar.shape[0], device=device)
    deg.index_add_(0, edge_index[0], torch.ones(edge_index.shape[1], device=device))
    deg.index_add_(0, edge_index[1], torch.ones(edge_index.shape[1], device=device))
    deg_ret = torch.zeros_like(deg)
    deg_ret.index_add_(0, edge_index[0, mask], torch.ones(mask.sum(), device=device))
    deg_ret.index_add_(0, edge_index[1, mask], torch.ones(mask.sum(), device=device))
    frac = deg_ret / deg.clamp(min=1)
    ke = 0.5 * torch.sum(frac * stellar * vd**2)
    u, v = edge_index[:, mask]
    r = torch.norm(pos[u] - pos[v], dim=1).clamp(min=1e-6)
    pe = G * torch.sum(stellar[u] * stellar[v] / r)
    return ke, pe


def relative_virial_penalty(ke_p, pe_p, ke_f, pe_f, eps_ratio=1e-12):
    # Approved Aug 2026: (log r_pruned - log r_full)^2 over loop-free physics
    # edges; r_full is the same graph's action-independent full-graph reference.
    # eps_ratio keeps zero-real-edge masks finite (maximal penalty band).
    def _r(ke, pe):
        pe = torch.clamp(torch.abs(torch.as_tensor(pe, dtype=torch.float32)), min=1e-30)
        return torch.clamp(2.0 * torch.as_tensor(ke, dtype=torch.float32) / pe, eps_ratio, 1.0/eps_ratio)
    return (torch.log(_r(ke_p, pe_p)) - torch.log(_r(ke_f, pe_f))) ** 2

def compute_rewards(pred_pruned, pred_full, y, keep_ratio, target_sp, virial_pen, cfg, conn_ok):
    base = torch.sqrt(((pred_full - y)**2).mean() + 1e-8)
    prun = torch.sqrt(((pred_pruned - y)**2).mean())
    delta = (base - prun) / base
    sp = ((keep_ratio - target_sp)**2).clamp(max=1.0)
    cb = torch.tensor(1.0 if conn_ok else -1.0)
    return (cfg["w_acc"]*delta - cfg["w_sp"]*sp + cfg["w_conn"]*cb
            - cfg["w_virial"]*virial_pen)


In [ ]:
# CELL 9: GNN adapter — full and pruned predictions for a graph dict
def gnns_adapter(g, mask):
    def run(edge_index, edge_attr):
        d = pg.Data(x=g["x"], edge_index=edge_index, edge_attr=edge_attr)
        d.batch = torch.zeros(d.x.shape[0], dtype=torch.long, device=device)
        with torch.no_grad():
            pred, _ = gnn(d)      # forward returns (predictions, embeddings)
            return pred.view(-1)
    mask = mask.to(device)
    pred_full = run(g["edge_index"], g["edge_attr"])
    pred_pruned = run(g["edge_index"][:, mask], g["edge_attr"][mask])
    return pred_full, pred_pruned


In [ ]:
# CELL 10: Policy-gradient training loop (Stage A) — ~20-30 min on T4
# REINFORCE + learned baseline (one-step MDP: no GAE, no importance ratio).
# GRADIENT FLOW (the bug class the review caught): the policy forward pass runs
# OUTSIDE torch.no_grad() so loss.backward() reaches policy.parameters().
# Only masks/GNN/rewards are under no_grad.
rls = cfg["rls"]
policy = EdgePolicyNet(edge_dim=cfg["graph"]["edge_features"].__len__(),
                       node_emb_dim=cfg["model"]["output_dim"],
                       hidden_dim=rls["policy_hidden"]).to(device)
value_net = ValueNet(cfg["model"]["output_dim"]).to(device)
opt = torch.optim.Adam(policy.parameters(), lr=rls["lr"])
vopt = torch.optim.Adam(value_net.parameters(), lr=rls["lr"])

def target_sparsity(epoch):
    p = min(1.0, epoch / max(1, rls["sparsity_anneal_epochs"]))
    return rls["target_sparsity_start"] + (rls["target_sparsity_end"] - rls["target_sparsity_start"]) * p

log = []
vrmse = float("inf")
for epoch in range(rls["epochs"]):
    ts = target_sparsity(epoch)
    order = torch.randperm(len(train_graphs)).tolist()
    ep_loss, ep_rewards = [], []
    for start in range(0, len(train_graphs), rls["batch_size"]):
        batch = [train_graphs[i] for i in order[start:start + rls["batch_size"]]]
        if not batch: continue
        logps, ents, vals, rews = [], [], [], []
        for g in batch:
            ctx = g["ctx"].to(device)
            # --- grad-carrying: policy + value net forwards ---
            probs = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                         g["edge_index"].to(device), ctx)).squeeze(-1)
            action = torch.bernoulli(probs)
            # logp on the RAW sampled action; the reward mask below uses the SAME
            # action (floored/repaired) — like action-clipping in continuous control.
            logps.append(bernoulli_logp(probs, action).mean())
            ents.append(bernoulli_entropy(probs).mean())
            v = value_net(ctx)
            vals.append(v)
            # --- no-grad: masks, GNN forwards, physics, rewards ---
            with torch.no_grad():
                hard = repair_connectivity(g["edge_index"].to(device),
                                           apply_min_keep_floor(action.bool(), probs,
                                                                rls["min_keep_frac"]))
                pf, pp = gnns_adapter(g, hard)
                ke_full, pe_full = graph_physics_terms(g, torch.ones(g["edge_index"].shape[1], dtype=torch.bool, device=device))
                ke, pe = graph_physics_terms(g, hard)
                vp = relative_virial_penalty(ke, pe, ke_full, pe_full) if rls["w_virial"] > 0 else torch.zeros(1, device=device)
                deg = torch.zeros(g["x"].shape[0], device=device)
                deg.index_add_(0, g["edge_index"][0, hard].to(device), torch.ones(hard.sum(), device=device))
                deg.index_add_(0, g["edge_index"][1, hard].to(device), torch.ones(hard.sum(), device=device))
                conn_ok = bool((hard.sum() > 0) and (deg >= 1).all())
                r = compute_rewards(pp, pf, g["y"].to(device), hard.float().mean(), ts, vp, rls, conn_ok)
            rews.append(r.view(1))
        logp = torch.stack(logps); ent = torch.stack(ents).mean()
        values = torch.stack(vals); rewards = torch.cat(rews)
        adv = compute_advantages(rewards, values)
        # policy update (REINFORCE + entropy)
        pg_total = -(adv.detach() * logp).mean() - rls["entropy_coef"] * ent
        opt.zero_grad(); pg_total.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
        # baseline (value net) update
        vf_loss = F.mse_loss(values, rewards.detach())
        vopt.zero_grad(); vf_loss.backward(); vopt.step()
        ep_loss.append((pg_total.detach() + vf_loss.detach()).item())
        ep_rewards.append(float(rewards.mean()))
    log.append([epoch, ts, float(np.mean(ep_loss)), float(np.mean(ep_rewards))])
    # validation: mean keep-fraction vs target, val MAE on pruned graphs
    if epoch % 10 == 0:
        keeps, rmses = [], []
        for g in val_graphs:
            with torch.no_grad():
                p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                         g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
                h = repair_connectivity(g["edge_index"].to(device), hard_mask(p, rls["min_keep_frac"]))
            keeps.append(h.float().mean().item())
            pf, pp = gnns_adapter(g, h)
            rmses.append((pp - g["y"].to(device)).abs().item())
        vrmse = float(np.mean(rmses))
        print(f"epoch {epoch}: loss={np.mean(ep_loss):.4f} reward={np.mean(ep_rewards):.3f} "
              f"keep={np.mean(keeps):.2f} (target {ts:.2f}) valMAE={vrmse:.4f}")
        torch.save(policy.state_dict(), "outputs/rls/policy.pt")
        torch.save(value_net.state_dict(), "outputs/rls/value_net.pt")
# Always save the FINAL weights: the periodic saves above stop at the last
# multiple of 10, so without this the checkpoint Notebooks C/D load (e.g. epoch
# 50) would silently differ from the weights this notebook verifies (epoch 59).
torch.save(policy.state_dict(), "outputs/rls/policy.pt")
torch.save(value_net.state_dict(), "outputs/rls/value_net.pt")
print("Stage A done. Last val MAE:", vrmse)
pd.DataFrame(log, columns=["epoch", "target_sp", "loss", "reward"]).to_csv("outputs/rls/training_log.csv", index=False)


> **PITFALL NOTE (see plan Part 4):** if the `keep` column stays pinned at the `min_keep_frac` floor for 5+ epochs, the policy is collapsing — raise `w_conn` to 2.0 and restart. If `valMAE` explodes (>0.25), the sparsity curriculum is too aggressive — slow the anneal (change `sparsity_anneal_epochs` to 50).

## Stage B (optional): fine-tune the GNN on the policy's pruned graphs

Skip this if you plan to run Notebook D (TTA) next — TTA needs the untouched frozen backbone.

In [ ]:
# CELL 11: Stage B — fine-tune GNN on policy-pruned graphs (closes distribution shift)
ft_graphs = train_graphs[:] 
masks = []
with torch.no_grad():
    for g in ft_graphs:
        p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                 g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
        masks.append(repair_connectivity(g["edge_index"].to(device), hard_mask(p, rls["min_keep_frac"])))

gnn.train()
ft_opt = torch.optim.AdamW(gnn.parameters(), lr=1e-4, weight_decay=5e-5)
for epoch in range(10):
    losses = []
    for g, m in zip(ft_graphs, masks):
        gx = pg.Data(x=g["x"], edge_index=g["edge_index"][:, m], edge_attr=g["edge_attr"][m])
        gx.batch = torch.zeros(gx.x.shape[0], dtype=torch.long, device=device)
        pred, _ = gnn(gx)
        loss = F.mse_loss(pred, g["y"].to(device).float())
        ft_opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(gnn.parameters(), 1.0); ft_opt.step()
        losses.append(loss.item())
    if epoch % 3 == 0:
        print(f"stageB epoch {epoch}: loss={np.mean(losses):.4f}")
torch.save(gnn.state_dict(), "outputs/rls/finetuned_gnn.pt")
print("Stage B done -> outputs/rls/finetuned_gnn.pt")


## Verify the trained policy on the test set

In [ ]:
# CELL 12: Verify final policy + fine-tuned GNN on test set
from model.physics_loss import MetricsComputer
preds_pol, preds_full, targets = [], [], []
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            n = g.x.shape[0]
            emb = gnn.get_embeddings(b, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, b.batch)
            p = torch.sigmoid(policy(g.edge_attr, emb[b.batch == i], g.edge_index, ctx[i])).squeeze(-1)
            m = repair_connectivity(g.edge_index, hard_mask(p, rls["min_keep_frac"]))
            d = pg.Data(x=g.x, edge_index=g.edge_index[:, m], edge_attr=g.edge_attr[m])
            d.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_pol, _ = gnn(d)
            preds_pol.append(p_pol.item()); targets.append(g.y.item())
            dfull = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr)
            dfull.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_full, _ = gnn(dfull)
            preds_full.append(p_full.item())
preds_pol = torch.tensor(preds_pol); preds_full = torch.tensor(preds_full); targets = torch.tensor(targets)
mp, mf = MetricsComputer.compute_all(preds_pol, targets), MetricsComputer.compute_all(preds_full, targets)
print(f"FULL  graph: RMSE={mf['rmse']:.4f} R2={mf['r2']:.4f}")
print(f"RL    pruned: RMSE={mp['rmse']:.4f} R2={mp['r2']:.4f}")
fid = np.corrcoef(preds_pol.numpy(), preds_full.numpy())[0, 1]
print(f"fidelity (Pearson): {fid:.4f}")


## Download your results

Grab `outputs/rls/policy.pt` (and `value_net.pt`) from Kaggle's output/session files, and add `policy.pt` to your `cosmicnet-data` Kaggle Dataset (Dataset -> Settings -> New Version -> upload) so Notebooks C and D can load it directly under `INPUT`. Also make sure Notebook A's `baselines.csv` is in that dataset — Notebook C's results table reads it, and it is not committed to the repo.

In [ ]:
# PROVENANCE (post-training): record the exact policy artifact B produced so
# C/D can verify they evaluate this very file (integrity check, not a new gate).
record_provenance("B_policy", extra={
    "policy_artifact": _file_info("outputs/rls/policy.pt"),
    "value_net_artifact": _file_info("outputs/rls/value_net.pt"),
})

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/rls_outputs", "zip", "outputs/rls")
print("Download /kaggle/working/rls_outputs.zip — contains policy.pt, value_net.pt, "
      "training_log.csv, and finetuned_gnn.pt if you ran Stage B")
